# 1b. BART-base baseline — fair retrain + clean-protocol evaluation

Replaces `1_bart_baseline_colab.ipynb` for the comparison table. Differences from the original,
each required for comparability with fusion_only / full_dual:

| | original (1) | this notebook (1b) | KG variants |
|---|---|---|---|
| data | re-downloads HF web_nlg | loads the SAME `webnlg_processed.pkl` from Drive | same pkl |
| epochs | 3 | 10, best checkpoint kept | 10 / 17, best kept |
| best-model criterion | dev BLEU (generate) | dev **loss** | dev val loss |
| objective | CE + label smoothing 0.1 | plain CE | plain CE |
| LR schedule | linear decay + 500 warmup | constant 3e-5, no warmup | constant 3e-5 |
| decoding | **beam-4** | **greedy** (argmax) | greedy |
| eval set | dev only, corrupted flattening | dedup test (2,510) + union refs + seen/unseen, plus dev | notebook-9 protocol |
| metric | old substring entity metric | fixed grounding metric + corruption taxonomy | notebook-9 metrics |

Training ≈ 10 × ~10 min on A100 + two decode passes (~25 min). Resume-safe.
**Send back:** everything in `kg_llm_project/baseline_v2_eval/`.


In [ ]:
# 1. Setup.
!pip -q install transformers datasets nltk rouge-score accelerate sentencepiece sacremoses sacrebleu

import os, sys, json, pickle, random, time, shutil
import numpy as np
import torch

from google.colab import drive
drive.mount('/content/drive')

print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# 2. Configuration.
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

PROJECT_DIR   = '/content/drive/MyDrive/kg_llm_project'
BASE_DATASET  = f'{PROJECT_DIR}/baseline-bart-webnlg'
PROCESSED_DIR = f'{BASE_DATASET}/processed'
BASELINE_V2   = f'{BASE_DATASET}/checkpoints/baseline_bart_v2_fair'   # new dir; old 3-epoch artifacts untouched
EVAL_OUT      = f'{PROJECT_DIR}/baseline_v2_eval'
for d in (BASELINE_V2, EVAL_OUT):
    os.makedirs(d, exist_ok=True)

MODEL_NAME     = 'facebook/bart-base'
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS         = 10          # same budget as fusion_only
BATCH_SIZE     = 32          # same as fusion_only
LR             = 3e-5        # same as the KG runs' bart_lr
MAX_INPUT_LEN  = 256
MAX_TARGET_LEN = 128
DONE_FLAG      = os.path.join(BASELINE_V2, 'training_done.flag')

print('BASELINE_V2:', BASELINE_V2)
print('EVAL_OUT:', EVAL_OUT)


In [ ]:
# 3. Load the SAME processed data the KG variants used. No re-download, no version drift.
req = [f'{PROCESSED_DIR}/webnlg_processed.pkl', f'{PROCESSED_DIR}/vocabularies.pkl']
missing = [p for p in req if not os.path.exists(p)]
if missing:
    raise FileNotFoundError('Missing processed artifacts (must be the originals from Drive): ' + str(missing))

with open(f'{PROCESSED_DIR}/webnlg_processed.pkl', 'rb') as f:
    data = pickle.load(f)
with open(f'{PROCESSED_DIR}/vocabularies.pkl', 'rb') as f:
    vocab = pickle.load(f)
print('splits:', {k: len(v) for k, v in data.items()})
assert len(data['train']) == 13211 and len(data['test']) == 5713, 'unexpected pkl — wrong file?'


In [ ]:
# 4. Train (plain CE, constant LR, best-by-dev-loss). Resume-safe; skipped if already done.
from transformers import (BartTokenizer, BartForConditionalGeneration,
                          DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments)
from transformers.trainer_utils import get_last_checkpoint
from datasets import Dataset

tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)

if os.path.exists(DONE_FLAG):
    print('Training already done — loading best model from', BASELINE_V2)
    model = BartForConditionalGeneration.from_pretrained(BASELINE_V2).to(DEVICE)
else:
    model = BartForConditionalGeneration.from_pretrained(MODEL_NAME)

    def to_hf(split):
        return Dataset.from_dict({
            'linearized': [ex['linearized'] for ex in split],
            'target': [ex['target'] for ex in split],
        })

    def preprocess(batch):
        enc = tokenizer(batch['linearized'], max_length=MAX_INPUT_LEN, truncation=True)
        lab = tokenizer(text_target=batch['target'], max_length=MAX_TARGET_LEN, truncation=True)
        enc['labels'] = lab['input_ids']
        return enc

    tok_train = to_hf(data['train']).map(preprocess, batched=True, remove_columns=['linearized', 'target'])
    tok_dev   = to_hf(data['dev']).map(preprocess, batched=True, remove_columns=['linearized', 'target'])
    collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

    args = Seq2SeqTrainingArguments(
        output_dir=BASELINE_V2,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        learning_rate=LR,
        lr_scheduler_type='constant',          # KG runs used constant-LR AdamW
        warmup_steps=0,
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        eval_strategy='epoch',                 # dev LOSS only — same selection signal as the KG runs
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        save_total_limit=2,
        logging_steps=200,
        label_smoothing_factor=0.0,            # plain CE (original had 0.1 — objective mismatch)
        predict_with_generate=False,
        report_to='none',
        seed=SEED,
    )
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=tok_train,
                             eval_dataset=tok_dev, data_collator=collator)
    last = get_last_checkpoint(BASELINE_V2)
    trainer.train(resume_from_checkpoint=last) if last else trainer.train()
    trainer.save_model(BASELINE_V2)
    tokenizer.save_pretrained(BASELINE_V2)
    with open(DONE_FLAG, 'w') as f:
        f.write('ok')
    model = trainer.model
    print('Saved fair baseline to', BASELINE_V2)

model.eval()
print('parameters:', sum(p.numel() for p in model.parameters()))


In [ ]:
# 5. Clean eval set — identical protocol to notebook 9 (dedup by ordered triples, union refs,
# seen/unseen = >=1 predicate absent from the 372 train predicates). Loads Drive index if present.
from collections import OrderedDict

def _triple_parts(t):
    if isinstance(t, dict):
        return str(t.get('subject', '')), str(t.get('predicate', '')), str(t.get('object', ''))
    return str(t[0]), str(t[1]), str(t[2])

def canonical_triple_key(triples):
    return json.dumps([_triple_parts(t) for t in triples], ensure_ascii=False)

def collect_references(example):
    refs = []
    for v in list(example.get('all_targets') or []) + [example.get('target', '')]:
        v = str(v).strip()
        if v and v not in refs:
            refs.append(v)
    return refs

groups = OrderedDict()
for i, ex in enumerate(data['test']):
    key = canonical_triple_key(ex['triples'])
    g = groups.setdefault(key, {'first_index': i, 'references': []})
    for r in collect_references(ex):
        if r not in g['references']:
            g['references'].append(r)

train_predicates = {_triple_parts(t)[1] for ex in data['train'] for t in ex['triples']}

unique_examples, unique_meta = [], []
for uid, (key, g) in enumerate(groups.items()):
    ex = dict(data['test'][g['first_index']])
    if not g['references']:
        raise ValueError(f'unique example {uid} has no references')
    ex['all_targets'] = g['references']
    preds_set = {_triple_parts(t)[1] for t in ex['triples']}
    unique_examples.append(ex)
    unique_meta.append({'unique_id': uid, 'key': key, 'first_original_index': g['first_index'],
                        'n_references': len(g['references']),
                        'split': 'unseen' if (preds_set - train_predicates) else 'seen'})

n_unseen = sum(m['split'] == 'unseen' for m in unique_meta)
print(f'{len(unique_examples)} unique inputs | seen {len(unique_examples)-n_unseen} / unseen {n_unseen}')
assert len(unique_examples) == 2510 and n_unseen == 752, 'dedup mismatch vs notebook 9 — investigate before continuing'


In [ ]:
# 6. Metrics — verbatim protocol from notebook 9 (fixed grounding metric + corruption taxonomy).
import re, unicodedata
from difflib import SequenceMatcher
import sacrebleu
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score as nltk_meteor
from rouge_score import rouge_scorer

for pkg in ('punkt', 'punkt_tab', 'wordnet', 'omw-1.4'):
    try: nltk.download(pkg, quiet=True)
    except Exception: pass
_rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

_TRANSLIT = {'ø':'o','Ø':'o','æ':'ae','Æ':'ae','œ':'oe','Œ':'oe','ð':'d','Ð':'d','þ':'th','Þ':'th',
             'ł':'l','Ł':'l','ß':'ss','đ':'d','Đ':'d','ħ':'h','ı':'i','İ':'i','ŋ':'ng'}
_MONTHS = ['january','february','march','april','may','june','july','august','september','october','november','december']
_DET = re.compile(r'^(the|a|an)\s+')
_STOP = frozenset(('the a an and or but if then this that these those it its he she they them his her their '
                   'in on at of to by with for from as is are was were be been being there here also however').split())

def _strip_accents(s):
    s = ''.join(_TRANSLIT.get(c, c) for c in str(s))
    return ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))

def _norm(s):
    s = _strip_accents(str(s)).lower().strip().strip('"').strip("'")
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return _DET.sub('', s)

def _word_match(text, surface):
    if not surface: return False
    return re.search(r'(?<![a-z0-9])' + re.escape(surface) + r'(?![a-z0-9])', text) is not None

def _date_surface_tokens(literal):
    toks = set(); raw = _strip_accents(str(literal)).strip().strip('"').strip("'")
    m = re.match(r'^(\d{3,4})-(\d{1,2})-(\d{1,2})$', raw)
    if m:
        y, mo, d = map(int, m.groups())
        toks.add(str(y))
        if 1 <= mo <= 12:
            toks.add(_MONTHS[mo-1]); toks.add(_MONTHS[mo-1][:3])
        toks.update({str(d), str(d).zfill(2)}); toks.update({f'{d}{s}' for s in ('st','nd','rd','th')})
    elif re.match(r'^\d{3,4}$', raw):
        toks.add(raw)
    return toks

def _digits(s): return re.sub(r'[^0-9]', '', str(s))
def _split_camel(s): return re.sub(r'([a-z])([A-Z])', r'\1 \2', str(s))

def _build_grounding(triples):
    forms, tokens, numcores = [], set(), set()
    for t in triples:
        s, p, o = _triple_parts(t)
        for v in (s, o):
            n = _norm(v)
            if n: forms.append(n); tokens.update(n.split())
            tokens.update(_date_surface_tokens(v))
            dg = _digits(v)
            if dg: numcores.add(dg)
        tokens.update(_norm(_split_camel(p)).split())
    return forms, tokens, numcores

def _extract_mentions(text):
    men = set()
    for m in re.findall(r'[A-Z][A-Za-z]*(?:[ -][A-Z][A-Za-z]*)*', _strip_accents(text)):
        n = _norm(m)
        if len(n) < 3: continue
        if ' ' not in n and n in _STOP: continue
        men.add(n)
    for m in re.findall(r'[A-Za-z0-9]+(?:[./\-][A-Za-z0-9]+)*', str(text)):
        if any(ch.isdigit() for ch in m): men.add(_norm(m))
    return {m for m in men if m}

def grounding_score(prediction, triples):
    pred_norm = _norm(prediction)
    forms, tokens, numcores = _build_grounding(triples)
    forms_despaced = [f.replace(' ', '') for f in forms]
    found = sum(1 for f in forms if _word_match(pred_norm, f) or all(_word_match(pred_norm, w) for w in f.split()))
    recall = found / len(forms) if forms else 1.0
    mentions = _extract_mentions(prediction)
    def grounded(m):
        for f in forms:
            if m == f or _word_match(f, m): return True
        if all(tok in tokens for tok in m.split()): return True
        dm = _digits(m)
        if dm and any(dm in c or c in dm for c in numcores): return True
        md = m.replace(' ', '')
        if len(md) >= 3 and any(md in x or x in md for x in forms_despaced): return True
        return False
    supported = {m for m in mentions if grounded(m)}
    hallucinated = sorted(mentions - supported)
    precision = len(supported) / len(mentions) if mentions else 1.0
    corruption, external = [], []
    for m in hallucinated:
        best = max((SequenceMatcher(None, m, f).ratio() for f in forms), default=0.0)
        (corruption if best >= 0.55 else external).append(m)
    return {'entity_precision': precision, 'entity_recall': recall,
            'hallucination_rate': 1.0 - precision, 'hallucinated_entities': hallucinated,
            'in_graph_corruptions': corruption, 'external_hallucinations': external}

def score_block(predictions, examples, metas):
    refs_list = [ex['all_targets'] for ex in examples]
    nltk_bleu = corpus_bleu([[r.split() for r in refs] for refs in refs_list],
                            [p.split() for p in predictions],
                            smoothing_function=SmoothingFunction().method1) * 100.0
    maxr = max(len(r) for r in refs_list)
    streams = [[refs[k] if k < len(refs) else refs[0] for refs in refs_list] for k in range(maxr)]
    sacre = sacrebleu.corpus_bleu(predictions, streams, lowercase=True, tokenize='13a').score
    rows, meteors, rouges = [], [], []
    for p, ex, m in zip(predictions, examples, metas):
        refs = ex['all_targets']
        try: meteors.append(float(nltk_meteor([r.split() for r in refs], p.split())))
        except Exception: meteors.append(0.0)
        rouges.append(max(_rouge.score(r, p)['rougeL'].fmeasure for r in refs))
        rows.append({**m, 'prediction': p, 'references': refs, **grounding_score(p, ex['triples'])})
    import pandas as pd
    frame = pd.DataFrame(rows)
    summary = {
        'n': len(predictions), 'nltk_corpus_bleu': float(nltk_bleu), 'sacrebleu_13a_lower': float(sacre),
        'meteor': float(np.mean(meteors)), 'rouge_l': float(np.mean(rouges)),
        'entity_precision': float(frame['entity_precision'].mean()),
        'entity_recall': float(frame['entity_recall'].mean()),
        'hallucination_rate': float(frame['hallucination_rate'].mean()),
        'examples_with_hallucination': float((frame['hallucination_rate'] > 0).mean()),
        'in_graph_corruption_mentions': int(frame['in_graph_corruptions'].map(len).sum()),
        'external_hallucination_mentions': int(frame['external_hallucinations'].map(len).sum()),
    }
    return summary, frame
print('metrics ready')


In [ ]:
# 7. GREEDY decode (num_beams=1 — matches the KG variants' argmax loop) on dedup test + dev.
@torch.no_grad()
def greedy_generate(texts, batch_size=32, tag=''):
    preds, t0 = [], time.time()
    for s in range(0, len(texts), batch_size):
        enc = tokenizer(texts[s:s+batch_size], max_length=MAX_INPUT_LEN, truncation=True,
                        padding=True, return_tensors='pt').to(DEVICE)
        out = model.generate(**enc, max_length=MAX_TARGET_LEN, num_beams=1, do_sample=False)
        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
        if (s // batch_size) % 10 == 9:
            print(f'  [{tag}] {s+batch_size}/{len(texts)} ({time.time()-t0:.0f}s)')
    return preds

TEST_PRED_PATH = os.path.join(EVAL_OUT, 'predictions_baseline_test.json')
if os.path.exists(TEST_PRED_PATH):
    test_preds = json.load(open(TEST_PRED_PATH))
    print('loaded existing test predictions:', len(test_preds))
else:
    test_preds = greedy_generate([ex['linearized'] for ex in unique_examples], tag='test')
    json.dump(test_preds, open(TEST_PRED_PATH, 'w'), ensure_ascii=False, indent=1)

DEV_PRED_PATH = os.path.join(EVAL_OUT, 'predictions_baseline_dev.json')
if os.path.exists(DEV_PRED_PATH):
    dev_preds = json.load(open(DEV_PRED_PATH))
else:
    dev_preds = greedy_generate([ex['linearized'] for ex in data['dev']], tag='dev')
    json.dump(dev_preds, open(DEV_PRED_PATH, 'w'), ensure_ascii=False, indent=1)
print('decoded: test', len(test_preds), '| dev', len(dev_preds))


In [ ]:
# 8. Score and save in the notebook-9 file format (baseline row for the comparison table).
import pandas as pd

all_summaries = []
for split_name in ('overall', 'seen', 'unseen'):
    idx = list(range(len(unique_examples))) if split_name == 'overall' else \
          [i for i, m in enumerate(unique_meta) if m['split'] == split_name]
    summary, frame = score_block([test_preds[i] for i in idx],
                                 [unique_examples[i] for i in idx],
                                 [unique_meta[i] for i in idx])
    all_summaries.append({'model': 'baseline_v2', 'alpha': 'n/a', 'split': split_name, **summary})
    fr = frame.copy()
    for col in ('references', 'hallucinated_entities', 'in_graph_corruptions', 'external_hallucinations'):
        fr[col] = fr[col].map(lambda v: json.dumps(v, ensure_ascii=False))
    fr.to_csv(os.path.join(EVAL_OUT, f'per_sample_baseline_{split_name}.csv'), index=False)

# dev block (all rows have refs; no dedup needed on dev)
dev_meta = [{'unique_id': i, 'key': '', 'first_original_index': i, 'n_references': len(ex.get('all_targets') or []), 'split': 'dev'}
            for i, ex in enumerate(data['dev'])]
dev_summary, _ = score_block(dev_preds, data['dev'], dev_meta)
all_summaries.append({'model': 'baseline_v2', 'alpha': 'n/a', 'split': 'dev', **dev_summary})

summary_frame = pd.DataFrame(all_summaries)
summary_frame.to_csv(os.path.join(EVAL_OUT, 'baseline_v2_summary.csv'), index=False)
display(summary_frame)

print('\nComparison anchors (fixed trie, same protocol):')
print('  full_dual alpha=0 : sacrebleu 47.39 | halluc 3.88% | recall 0.8024  (overall)')
print('  fusion_only(broken trie, ~= alpha 0): sacrebleu ~47.3 | halluc 3.79% | recall 0.8000')
print('\nSEND BACK:', EVAL_OUT)
for f in sorted(os.listdir(EVAL_OUT)): print('  ', f)
